# GVH–CHH · A1 · Deux compartiments de gaz
**CHH-COMP-01 · Cas 02 · v0.1 · Brouillon pré-gel**  
Protocole CHH v1.0 · [Cas 01, deux masses](./CHH_COMP_01_Deux_Masses_A1_v01.ipynb)

## Niveau 1 — CHH-STRUCTURE et INVERSE-INFERENCE

Deux compartiments rigides de volumes connus $V_1,V_2>0$, en litres, contiennent le même gaz idéal à la même température absolue $T>0$. L'enceinte est fermée aux échanges de matière. Après mise en communication, le gaz atteint un équilibre à la même température $T$, maintenue par un thermostat. Les pressions absolues initiales $p_1,p_2$ et finale $p_f$ sont exprimées en kPa.

**Question directe.** Calculer $p_f$ à partir des pressions initiales.  
**Question inverse.** Déterminer si $p_f$ suffit, puis ce que change une mesure de $p_1$ avant ouverture. La cible est $p_2$. La CHH décrit ici l'agrégation de deux compartiments. Le calcul est statique.

## Niveau 2 — KNOWN-PHYSICS

La conservation du nombre total de moles et la loi du gaz parfait donnent
$$p_f(V_1+V_2)=p_1V_1+p_2V_2,\qquad
p_f=(1-w)p_1+wp_2,\qquad w=\frac{V_2}{V_1+V_2}.$$
Le produit kPa·L vaut un joule. Le poids $w$ est sans dimension. Le jacobien de $p_f$ par rapport aux deux pressions est la ligne $(1-w,w)$, de rang 1. Avec la mesure supplémentaire $p_1$, le jacobien de $(p_1,p_f)$ a pour déterminant $w$. Si $w>0$, on obtient
$$\widehat p_2=\frac{p_f^{\rm mes}-(1-w)p_1^{\rm mes}}{w}.$$

Avec des bruits additifs indépendants centrés, d'écarts-types $\sigma_f$ et $\sigma_1$, l'erreur est $(\epsilon_f-(1-w)\epsilon_1)/w$. L'écart-type prédit est
$$\sigma_{p_2}=\frac{\sqrt{\sigma_f^2+(1-w)^2\sigma_1^2}}{w}.$$
Si $p_1$ est exacte, il devient $\sigma_f/w$. À $w=0$, le modèle limite perd toute sensibilité à $p_2$. Ce cas sert de contrôle mathématique.

## Niveau 3 — NUMERICAL-TEST

Plan de simulation. Générer 10 000 couples de pressions uniformes indépendantes entre 80 et 120 kPa. Fixer le volume total à 10 L et tester $w=0{,}50$, $0{,}20$, $0{,}05$, $0{,}01$, en réutilisant les mêmes pressions et tirages de bruit pour comparer les géométries. Les bruits gaussiens de $p_f$ et $p_1$ ont chacun un écart-type de 0,20 kPa. Les volumes sont exacts. La formule d'inversion est fixée avant la génération des données et reçoit uniquement les observations et le poids. Les vérités synthétiques servent au calcul des erreurs.

Contrôles fixés dans ce notebook. Vérifier deux états initiaux distincts donnant le même $p_f$, la reconstruction exacte à $10^{-10}$ kPa près, la dégénérescence limite à $w=0$, le facteur 50 d'amplification quand seul $p_f$ est bruité aux deux poids extrêmes, et un écart relatif inférieur à 4 % entre RMSE simulée et écart-type théorique quand les deux mesures sont bruitées. Les sorties hors du domaine physique sont comptées et conservées dans la RMSE.

In [1]:
from math import isclose, sqrt
from random import Random
from statistics import fmean
import platform

SEED = 20261120
N = 10_000
TOTAL_VOLUME = 10.0  # L
SIGMA_FINAL = 0.20  # kPa
SIGMA_INITIAL = 0.20  # kPa
FRACTIONS = (0.50, 0.20, 0.05, 0.01)

def pression_finale(p1, p2, fraction):
    return (1 - fraction) * p1 + fraction * p2

def pression_initiale_2(p1_mesuree, pf_mesuree, fraction):
    if fraction == 0:
        return None
    return (pf_mesuree - (1 - fraction) * p1_mesuree) / fraction

print(f"Python {platform.python_version()} | graine {SEED} | {N} systèmes par géométrie")
assert pression_finale(90, 110, 0.5) == pression_finale(95, 105, 0.5) == 100
assert pression_finale(90, 110, 0) == pression_finale(90, 95, 0) == 90
assert pression_initiale_2(90, 90, 0) is None
print("PASS · Pression finale seule ambiguë et limite w = 0 dégénérée.")

Python 3.12.14 | graine 20261120 | 10000 systèmes par géométrie
PASS · Pression finale seule ambiguë et limite w = 0 dégénérée.


In [2]:
rng = Random(SEED)
pressions = [(rng.uniform(80, 120), rng.uniform(80, 120)) for _ in range(N)]
bruits = [(rng.gauss(0, SIGMA_INITIAL), rng.gauss(0, SIGMA_FINAL)) for _ in range(N)]

resultats = []
print("V2 (L) | w     | RMSE pf seule* | RMSE deux mesures* | théorie deux mesures* | p2 ≤ 0")
for w in FRACTIONS:
    erreur_max = 0.0
    erreurs_finales, erreurs_deux = [], []
    hors_domaine = 0
    for (p1, p2), (eps1, epsf) in zip(pressions, bruits, strict=True):
        pf = pression_finale(p1, p2, w)
        erreur_max = max(erreur_max, abs(pression_initiale_2(p1, pf, w) - p2))
        erreurs_finales.append(pression_initiale_2(p1, pf + epsf, w) - p2)
        p2_estimee = pression_initiale_2(p1 + eps1, pf + epsf, w)
        erreurs_deux.append(p2_estimee - p2)
        hors_domaine += p2_estimee <= 0
    rmse_finale = sqrt(fmean(e**2 for e in erreurs_finales))
    rmse_deux = sqrt(fmean(e**2 for e in erreurs_deux))
    theorie = sqrt(SIGMA_FINAL**2 + (1 - w)**2 * SIGMA_INITIAL**2) / w
    assert erreur_max < 1e-10
    assert isclose(rmse_deux, theorie, rel_tol=0.04)
    resultats.append((w, rmse_finale, rmse_deux, theorie, erreur_max))
    print(f"{TOTAL_VOLUME*w:6.2f} | {w:.3f} | {rmse_finale:15.3f} | {rmse_deux:19.3f} | {theorie:22.3f} | {hors_domaine}")
assert isclose(resultats[-1][1] / resultats[0][1], 50, rel_tol=1e-12)
print("* kPa. La colonne « pf seule » signifie que seul pf est bruité et que p1 est exacte.")
print(f"Erreur maximale sans bruit : {max(row[4] for row in resultats):.3e} kPa")
print("Verdict : PASS_SYNTHETIC_TWO_GAS_COMPARTMENTS_IDENTIFIABILITY")

V2 (L) | w     | RMSE pf seule* | RMSE deux mesures* | théorie deux mesures* | p2 ≤ 0
  5.00 | 0.500 |           0.403 |               0.449 |                  0.447 | 0
  2.00 | 0.200 |           1.006 |               1.283 |                  1.281 | 0
  0.50 | 0.050 |           4.026 |               5.523 |                  5.517 | 0
  0.10 | 0.010 |          20.128 |              28.168 |                 28.143 | 1
* kPa. La colonne « pf seule » signifie que seul pf est bruité et que p1 est exacte.
Erreur maximale sans bruit : 7.105e-13 kPa
Verdict : PASS_SYNTHETIC_TWO_GAS_COMPARTMENTS_IDENTIFIABILITY


## Niveau 4 — Verdict et portée

La pression finale seule fournit une combinaison pondérée des pressions initiales. Une mesure de $p_1$ permet d'identifier $p_2$ pour tout volume cible strictement positif sous les hypothèses du modèle. À précision de mesure fixée, plus la fraction volumique cible est faible, plus l'inversion amplifie le bruit. Le cas des deux masses montre un mécanisme analogue lorsque les positions se rapprochent.

**Portée du PASS.** Loi du gaz idéal à température commune fixée, volumes connus et données synthétiques. Aucune phase transitoire ni fuite n'est modélisée. Une extension sur mesures physiques devrait préciser la stabilité de la température, l'incertitude des volumes et la covariance éventuelle des capteurs. A2 (INVERSE-03B) reste l'étape de validation sur catalogue astronomique.